In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dansbecker/powerlifting-database/meets.csv
/kaggle/input/datasets/dansbecker/powerlifting-database/openpowerlifting.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("ETL_pipeline").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/17 04:59:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
meet_url ="meetpath"
power_url = "powerpath"

Reading the data

In [9]:
import pandas as pd
meet_df = pd.read_csv(meet_url)
power_df = pd.read_csv(power_url)

In [10]:
print(meet_df)

      MeetID                   MeetPath Federation        Date MeetCountry  \
0          0             365strong/1601  365Strong  2016-10-29         USA   
1          1             365strong/1602  365Strong  2016-11-19         USA   
2          2             365strong/1603  365Strong  2016-07-09         USA   
3          3             365strong/1604  365Strong  2016-06-11         USA   
4          4             365strong/1605  365Strong  2016-04-10         USA   
...      ...                        ...        ...         ...         ...   
8477    8477            xpc/2015-finals        XPC  2015-03-06         USA   
8478    8478  xpc/2016-bench-freak-show        XPC  2016-03-04         USA   
8479    8479      xpc/2016-elite-finals        XPC  2016-03-04         USA   
8480    8480        xpc/2016-pro-finals        XPC  2016-03-05         USA   
8481    8481            xpc/2017-finals        XPC  2017-03-03         USA   

     MeetState   MeetTown                                      

In [11]:
print(power_df)

        MeetID              Name Sex   Equipment   Age     Division  \
0            0  Angie Belk Terry   F       Wraps  47.0    Mst 45-49   
1            0       Dawn Bogart   F  Single-ply  42.0    Mst 40-44   
2            0       Dawn Bogart   F  Single-ply  42.0  Open Senior   
3            0       Dawn Bogart   F         Raw  42.0  Open Senior   
4            0      Destiny Dula   F         Raw  18.0   Teen 18-19   
...        ...               ...  ..         ...   ...          ...   
386409    8481   William Barabas   M   Multi-ply   NaN        Elite   
386410    8481      Justin Zottl   M   Multi-ply   NaN        Elite   
386411    8481     Jake Anderson   M   Multi-ply   NaN        Elite   
386412    8481    Jeff Bumanglag   M   Multi-ply   NaN        Elite   
386413    8481     Shane Hammock   M   Multi-ply   NaN        Elite   

        BodyweightKg WeightClassKg  Squat4Kg  BestSquatKg  Bench4Kg  \
0              59.60            60       NaN        47.63       NaN   
1    

In [12]:
power_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386414 entries, 0 to 386413
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   MeetID          386414 non-null  int64  
 1   Name            386414 non-null  object 
 2   Sex             386414 non-null  object 
 3   Equipment       386414 non-null  object 
 4   Age             147147 non-null  float64
 5   Division        370571 non-null  object 
 6   BodyweightKg    384012 non-null  float64
 7   WeightClassKg   382602 non-null  object 
 8   Squat4Kg        1243 non-null    float64
 9   BestSquatKg     298071 non-null  float64
 10  Bench4Kg        1962 non-null    float64
 11  BestBenchKg     356364 non-null  float64
 12  Deadlift4Kg     2800 non-null    float64
 13  BestDeadliftKg  317847 non-null  float64
 14  TotalKg         363237 non-null  float64
 15  Place           385322 non-null  object 
 16  Wilks           362194 non-null  float64
dtypes: float64

In [13]:
power_df.isna().sum()

MeetID                 0
Name                   0
Sex                    0
Equipment              0
Age               239267
Division           15843
BodyweightKg        2402
WeightClassKg       3812
Squat4Kg          385171
BestSquatKg        88343
Bench4Kg          384452
BestBenchKg        30050
Deadlift4Kg       383614
BestDeadliftKg     68567
TotalKg            23177
Place               1092
Wilks              24220
dtype: int64

In [14]:
mdf = spark.createDataFrame(meet_df)
mdf.show

<bound method DataFrame.show of DataFrame[MeetID: bigint, MeetPath: string, Federation: string, Date: string, MeetCountry: string, MeetState: string, MeetTown: string, MeetName: string]>

In [15]:
pdf = spark.createDataFrame(power_df)
pdf.show

<bound method DataFrame.show of DataFrame[MeetID: bigint, Name: string, Sex: string, Equipment: string, Age: double, Division: string, BodyweightKg: double, WeightClassKg: string, Squat4Kg: double, BestSquatKg: double, Bench4Kg: double, BestBenchKg: double, Deadlift4Kg: double, BestDeadliftKg: double, TotalKg: double, Place: string, Wilks: double]>

Extraction complete. Transformation

Data cleaning

In [16]:
pdf.printSchema()

root
 |-- MeetID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Equipment: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Division: string (nullable = true)
 |-- BodyweightKg: double (nullable = true)
 |-- WeightClassKg: string (nullable = true)
 |-- Squat4Kg: double (nullable = true)
 |-- BestSquatKg: double (nullable = true)
 |-- Bench4Kg: double (nullable = true)
 |-- BestBenchKg: double (nullable = true)
 |-- Deadlift4Kg: double (nullable = true)
 |-- BestDeadliftKg: double (nullable = true)
 |-- TotalKg: double (nullable = true)
 |-- Place: string (nullable = true)
 |-- Wilks: double (nullable = true)



In [17]:
pdf.describe().show()

26/07/17 05:03:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/17 05:03:34 WARN TaskSetManager: Stage 0 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


+-------+------------------+------------+------+---------+------+----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+------+------+
|summary|            MeetID|        Name|   Sex|Equipment|   Age|  Division|BodyweightKg|WeightClassKg|Squat4Kg|BestSquatKg|Bench4Kg|BestBenchKg|Deadlift4Kg|BestDeadliftKg|TotalKg| Place| Wilks|
+-------+------------------+------------+------+---------+------+----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+------+------+
|  count|            386414|      386414|386414|   386414|386414|    386414|      386414|       386414|  386414|     386414|  386414|     386414|     386414|        386414| 386414|386414|386414|
|   mean| 5143.015804292805|        NULL|  NULL|     NULL|   NaN|       NaN|         NaN|          NaN|     NaN|        NaN|     NaN|        NaN|        NaN|           NaN|    NaN|   NaN|   NaN|
| stddev|2552.09983814424

In [18]:
pdf.show()

26/07/17 05:03:55 WARN TaskSetManager: Stage 3 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


+------+--------------------+---+----------+----+-----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+
|MeetID|                Name|Sex| Equipment| Age|   Division|BodyweightKg|WeightClassKg|Squat4Kg|BestSquatKg|Bench4Kg|BestBenchKg|Deadlift4Kg|BestDeadliftKg|TotalKg|Place| Wilks|
+------+--------------------+---+----------+----+-----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+
|     0|    Angie Belk Terry|  F|     Wraps|47.0|  Mst 45-49|        59.6|           60|     NaN|      47.63|     NaN|      20.41|        NaN|         70.31| 138.35|    1|155.05|
|     0|         Dawn Bogart|  F|Single-ply|42.0|  Mst 40-44|       58.51|           60|     NaN|     142.88|     NaN|      95.25|        NaN|        163.29| 401.42|    1|456.38|
|     0|         Dawn Bogart|  F|Single-ply|42.0|Open Senior|       58.51|           60|     NaN|     142

In [20]:
pdf = pdf.fillna({"Age":0})

In [21]:
pdf.show()

26/07/17 05:04:11 WARN TaskSetManager: Stage 4 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


+------+--------------------+---+----------+----+-----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+
|MeetID|                Name|Sex| Equipment| Age|   Division|BodyweightKg|WeightClassKg|Squat4Kg|BestSquatKg|Bench4Kg|BestBenchKg|Deadlift4Kg|BestDeadliftKg|TotalKg|Place| Wilks|
+------+--------------------+---+----------+----+-----------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+
|     0|    Angie Belk Terry|  F|     Wraps|47.0|  Mst 45-49|        59.6|           60|     NaN|      47.63|     NaN|      20.41|        NaN|         70.31| 138.35|    1|155.05|
|     0|         Dawn Bogart|  F|Single-ply|42.0|  Mst 40-44|       58.51|           60|     NaN|     142.88|     NaN|      95.25|        NaN|        163.29| 401.42|    1|456.38|
|     0|         Dawn Bogart|  F|Single-ply|42.0|Open Senior|       58.51|           60|     NaN|     142

In [28]:
from pyspark.sql.functions import col,isnan
pdf.filter(isnan(col("Squat4Kg"))).count()

26/07/17 05:13:42 WARN TaskSetManager: Stage 6 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


385171

In [29]:
pdf.filter(isnan(col("Age"))).count()

26/07/17 05:14:30 WARN TaskSetManager: Stage 9 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


0

In [32]:
pdf = pdf.fillna({"Squat4Kg":0})

In [35]:
mergedf = pdf.join(mdf,on='MeetID', how='inner')
mergedf.show()

26/07/17 05:22:59 WARN TaskSetManager: Stage 20 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


+------+--------------------+---+---------+---+--------------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+--------+----------+----------+-----------+---------+---------+--------------------+
|MeetID|                Name|Sex|Equipment|Age|      Division|BodyweightKg|WeightClassKg|Squat4Kg|BestSquatKg|Bench4Kg|BestBenchKg|Deadlift4Kg|BestDeadliftKg|TotalKg|Place| Wilks|MeetPath|Federation|      Date|MeetCountry|MeetState| MeetTown|            MeetName|
+------+--------------------+---+---------+---+--------------+------------+-------------+--------+-----------+--------+-----------+-----------+--------------+-------+-----+------+--------+----------+----------+-----------+---------+---------+--------------------+
|    26|     Michele Sosnine|  F|    Wraps|0.0|   DT - Junior|       47.54|           48|     0.0|      100.0|     NaN|       57.5|        NaN|          87.5|  245.0|    1|326.78|apa/1602|       APA|2016-04-1

In [66]:
#aggdf = (mergedf.filter("Sex = 'F'").groupby("MeetState","Sex")\
    #     .count().alias("Female_count").orderBy(col("Female_count").desc()))

#         .orderby(col("Female_count").desc()))

aggdf = (
    mergedf
    .filter(col("Sex") == "F").filter(col("MeetState") != "NaN")
    .groupBy("MeetState", "Sex")
    .count()
    .withColumnRenamed("count", "Female_count")
    .orderBy(col("Female_count").desc())
)
aggdf.show()

26/07/17 06:36:46 WARN TaskSetManager: Stage 79 contains a task of very large size (11422 KiB). The maximum recommended task size is 1000 KiB.


+---------+---+------------+
|MeetState|Sex|Female_count|
+---------+---+------------+
|       CA|  F|        7437|
|       TX|  F|        5893|
|       FL|  F|        3511|
|       ON|  F|        2896|
|       NY|  F|        2593|
|       OH|  F|        2586|
|       PA|  F|        2253|
|       NV|  F|        2068|
|       GA|  F|        1992|
|       WI|  F|        1976|
|       WA|  F|        1680|
|       CO|  F|        1650|
|      VIC|  F|        1603|
|       IL|  F|        1540|
|       NJ|  F|        1450|
|       BC|  F|        1195|
|      NSW|  F|        1191|
|       MA|  F|        1181|
|       AB|  F|        1148|
|       AZ|  F|        1124|
+---------+---+------------+
only showing top 20 rows
